<a href="https://colab.research.google.com/github/ShivaBandereddy/incident-triage-agent/blob/main/day2_langgraph_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
import os
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

In [2]:
!pip install -q langgraph langchain-google-genai

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.3.0 which is incompatible.


In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class S(TypedDict):
    text: str
    score: int

def a(state: S):
    return {'text': state['text'] + ' world', 'score': len(state['text'])}

def b(state: S):
    return {'text': state['text'].upper()}

def route(state: S):
    return 'b' if state['score'] > 3 else END

g = StateGraph(S)
g.add_node('a', a)
g.add_node('b', b)
g.add_edge(START, 'a')
g.add_conditional_edges('a', route, {'b': 'b', END: END})
g.add_edge('b', END)
app = g.compile()

print(app.invoke({'text': 'hi', 'score': 0}))

{'text': 'hi world', 'score': 2}


In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite', timeout=30, max_retries=1)

class S2(TypedDict):
    alert_text: str
    summary: str

def summarize(state: S2):
    msg = 'Summarize this alert in one short line: ' + state['alert_text']
    res = llm.invoke(msg)
    text = res.content
    if isinstance(text, list):
        text = ''.join(p.get('text', '') for p in text if isinstance(p, dict))
    return {'summary': text}

g2 = StateGraph(S2)
g2.add_node('summarize', summarize)
g2.add_edge(START, 'summarize')
g2.add_edge('summarize', END)
app2 = g2.compile()

out = app2.invoke({'alert_text': 'ALERT: Datastore ds-prod-01 read latency above 100 ms for 10 min', 'summary': ''})
print(out['summary'])

Datastore ds-prod-01 has high read latency (>100 ms) for 10 minutes.


In [7]:
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI

class Triage(BaseModel):
    service: str = Field(description='affected service or component')
    severity: str = Field(description='low, medium, high or critical')

llm = ChatGoogleGenerativeAI(model='gemini-3.5-flash-lite', timeout=30, max_retries=1)
structured = llm.with_structured_output(Triage)

result = structured.invoke('ALERT: Host esx-host-03 not responding')
print(result.service, '|', result.severity)
print(type(result))
print(result.service, '|', result.severity)

esx-host-03 | critical
<class '__main__.Triage'>
esx-host-03 | critical
